In [11]:
import pandas as pd
import json

from pymongo import MongoClient

In [13]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["censo_locales_db"]
collection = db["locales"]

a. Índice simple: crea un índice sobre un campo único, como el nombre del
barrio. Asegúrate de que la búsqueda de documentos en base a este campo
sea más eficiente.

• Pista: Utiliza el comando db.collection.createIndex({ campo: 1
}) para crear un índice en orden ascendente. Luego, ejecuta una
consulta que filtre los datos por barrio para observar el impacto del
índice en el rendimiento.


In [79]:
collection.create_index(
    [("local.desc_barrio_local", 1)],
    name="idx_barrio"
)

'idx_barrio'

In [57]:
collection.drop_index("idx_barrio")

In [53]:
filtro = {
    "local.desc_barrio_local": {
        "$regex": "^ACACIAS",
        "$options": "i"
    }
}

resultado = list(collection.find(filtro))
print(len(resultado))

1236


In [93]:
plan = collection.find(filtro).explain()

print("\n" + "="*50)
print("📊 RESULTADO EXPLAIN")
print("="*50)

print(f"Tiempo de ejecución: {plan['executionStats']['executionTimeMillis']} ms")
print(f"Documentos examinados: {plan['executionStats']['totalDocsExamined']}")
print(f"Claves de índice examinadas: {plan['executionStats']['totalKeysExamined']}")

print("\nPlan de ejecución ganador:")
print("-"*50)

from pprint import pformat
print(pformat(plan["queryPlanner"]["winningPlan"], indent=2))




📊 RESULTADO EXPLAIN
Tiempo de ejecución: 120 ms
Documentos examinados: 1236
Claves de índice examinadas: 151162

Plan de ejecución ganador:
--------------------------------------------------
{ 'inputStage': { 'direction': 'forward',
                  'filter': { 'local.desc_barrio_local': { '$options': 'i',
                                                           '$regex': '^ACACIAS'}},
                  'indexBounds': { 'local.desc_barrio_local': [ '["", {})',
                                                                '[/^ACACIAS/i, '
                                                                '/^ACACIAS/i]']},
                  'indexName': 'idx_barrio',
                  'indexVersion': 2,
                  'isMultiKey': False,
                  'isPartial': False,
                  'isSparse': False,
                  'isUnique': False,
                  'keyPattern': {'local.desc_barrio_local': 1},
                  'multiKeyPaths': {'local.desc_barrio_local': []},
   

b. Índice compuesto: diseña un índice compuesto que combine los campos
distrito y barrio. Este índice debe mejorar el rendimiento en consultas que
incluyan ambos campos.
• Pista: usa el comando db.collection.createIndex({ campo1: 1,
campo2: 1 }). Recuerda que el orden de los campos en un índice
compuesto es importante, así que considera cómo suelen ser tus
consultas para decidir el orden adecuado.

In [113]:
collection.create_index(
    [
        ("local.desc_distrito_local", 1),
        ("local.desc_barrio_local", 1)
    ],
    name="idx_distrito_barrio"
)

'idx_distrito_barrio'

In [103]:
collection.drop_index("idx_distrito_barrio")

In [99]:
filtro = {
    "local.desc_distrito_local": {
        "$regex": "^SALAMANCA",
        "$options": "i"
    },
    "local.desc_barrio_local": {
        "$regex": "^GUINDALERA",
        "$options": "i"
    }
}


resultado = list(collection.find(filtro))
print(len(resultado))

1825


In [114]:
plan = collection.find(filtro).explain()

print("\n" + "="*50)
print("📊 RESULTADO EXPLAIN")
print("="*50)

print(f"Tiempo de ejecución: {plan['executionStats']['executionTimeMillis']} ms")
print(f"Documentos examinados: {plan['executionStats']['totalDocsExamined']}")
print(f"Claves de índice examinadas: {plan['executionStats']['totalKeysExamined']}")

print("\nPlan de ejecución ganador:")
print("-"*50)

from pprint import pformat
print(pformat(plan["queryPlanner"]["winningPlan"], indent=2))


📊 RESULTADO EXPLAIN
Tiempo de ejecución: 138 ms
Documentos examinados: 1825
Claves de índice examinadas: 151162

Plan de ejecución ganador:
--------------------------------------------------
{ 'inputStage': { 'direction': 'forward',
                  'filter': { '$and': [ { 'local.desc_distrito_local': { '$options': 'i',
                                                                         '$regex': '^SALAMANCA'}},
                                        { 'local.desc_barrio_local': { '$options': 'i',
                                                                       '$regex': '^GUINDALERA'}}]},
                  'indexBounds': { 'local.desc_barrio_local': [ '["", {})',
                                                                '[/^GUINDALERA/i, '
                                                                '/^GUINDALERA/i]'],
                                   'local.desc_distrito_local': [ '["", {})',
                                                                  '

Índice de array: si tienes un campo que almacena un array (por ejemplo, una
lista de actividades económicas asociadas a un local o terraza), crea un índice
sobre ese campo para permitir búsquedas rápidas de documentos que
contengan valores específicos en el array.
8

• Pista: usa el comando db.collection.createIndex({ campo: 1 }).
Luego, realiza una consulta para buscar documentos que incluyan una
actividad económica específica, como { actividades_economicas:
"Restauración" }.


In [154]:
filtro = {
    "actividadeconomica.desc_barrio_local": {
        "$regex": "^ARCOS",
        "$options": "i"
    }
}

resultado = list(collection.find(filtro))
print(len(resultado))

400


In [155]:
collection.create_index(
    [("actividadeconomica.desc_barrio_local", 1)],
    name="idx_actividad"
)


'idx_actividad'

In [149]:
collection.drop_index("idx_actividad")

In [157]:
plan = collection.find(filtro).explain()

print("\n" + "="*50)
print("📊 RESULTADO EXPLAIN")
print("="*50)

print(f"Tiempo de ejecución: {plan['executionStats']['executionTimeMillis']} ms")
print(f"Documentos examinados: {plan['executionStats']['totalDocsExamined']}")
print(f"Claves de índice examinadas: {plan['executionStats']['totalKeysExamined']}")

print("\nPlan de ejecución ganador:")
print("-"*50)

from pprint import pformat
print(pformat(plan["queryPlanner"]["winningPlan"], indent=2))


📊 RESULTADO EXPLAIN
Tiempo de ejecución: 472 ms
Documentos examinados: 151161
Claves de índice examinadas: 151161

Plan de ejecución ganador:
--------------------------------------------------
{ 'filter': { 'actividadeconomica.desc_barrio_local': { '$options': 'i',
                                                        '$regex': '^ARCOS'}},
  'inputStage': { 'direction': 'forward',
                  'indexBounds': { 'actividadeconomica.desc_barrio_local': [ '["", '
                                                                             '{})',
                                                                             '[/^ARCOS/i, '
                                                                             '/^ARCOS/i]']},
                  'indexName': 'idx_actividad',
                  'indexVersion': 2,
                  'isMultiKey': True,
                  'isPartial': False,
                  'isSparse': False,
                  'isUnique': False,
                  'keyPa